In [1]:
import requests
import os
import pandas as pd
import time
from datetime import datetime, timedelta
import gzip
import shutil

base_url = os.environ['CD2_BASE_URL']
client_id = os.environ['CD2_CLIENT_ID']
client_secret = os.environ['CD2_CLIENT_SECRET']

In [2]:
def hent_filar_web_logs(innfil, n):
    """
    Hentar datafil frå CD2 og pakkar den ut.
    Argument:
    tabell - namnet på tabellen som skal hentast
    innfil - id på fila som skal hentast
    n - rekkjefølgje på fila som skal hentast (for logging)
    Returnerer namnet på den utpakkede fila.
    """
    requesturl = f"{base_url}/dap/object/url"
    payload = f"{respons2['objects']}"
    payload = payload.replace('\'', '\"')
    headers = {
        'x-instauth': access_token, 
        'Content-Type': 'text/plain'
        }
    print(f"Hentar datafil nr. {n}, ", end="")
    r4 = requests.post(
        requesturl, 
        headers=headers, 
        data=payload
        )
    if r4.status_code == 200:
        respons4 = r4.json()
        url = respons4['urls'][innfil]['url']
        data = requests.request("GET", url)
        no = datetime.now()
        utfil = f"web_logs-{no.year}{no.month:02}{no.day:02}{no.hour:02}{no.minute:02}-{n}"
        open(f'{utfil}.gz', 'wb').write(data.content)
        with gzip.open(f'{utfil}.gz', 'rb') as f_in:
            with open(f'{utfil}.txt', 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print(f" skrevet til {f'{utfil}.txt'}")
        os.remove(f"{utfil}.gz")
    return f"{utfil}.txt"

# 0. Set opp systemet
Først må eg hente token for tilgang

In [3]:
auth_url = f"{base_url}/ids/auth/login"
payload={'grant_type': 'client_credentials'}
r = requests.post(
    auth_url, 
    data=payload, 
    auth=(client_id, client_secret))
if 200 <= r.status_code < 300:
    respons = r.json()
    access_token = respons['access_token']
    print("Henta access_token OK")
else:
    print(f"Klarte ikkje å skaffe access_token, feil {r.status_code}")

Henta access_token OK


# 1. Hente data
Data eg henter her er frå den "store" loggen **web_logs**. Den har eit eige endepunkt (`{base_url}/dap/query/canvas_logs/table/web_logs/data`)

In [ ]:
no = datetime.now()
timar = 4
tidsrom = timedelta(hours=timar)
sist_oppdatert = (no - tidsrom).isoformat(timespec='seconds') + "Z"
requesturl = f"{base_url}/dap/query/canvas_logs/table/web_logs/data"
payload = '{"format": "csv", "since": \"%s\"}' %(sist_oppdatert)
headers = {'x-instauth': access_token, 'Content-Type': 'text/plain'}
try:
    print(f"Sender inkrementell spørjing for dei siste {timar} timane til {requesturl}")
    r = requests.post(
        requesturl, 
        headers=headers, 
        data=payload
        )
    if 200 <= r.status_code < 300:
        respons = r.json()
        id = respons['id']
        les_data = True
        while les_data:
            print(f"Sjekker status på jobb {id}")
            requesturl = f"{base_url}/dap//job/{id}"
            r2 = requests.get(requesturl, headers=headers)
            if 200 <= r2.status_code < 300:
                respons2 = r2.json()
                print(respons2)
                if respons2['status'] == "complete":
                    les_data = False
                time.sleep(5)
        antal = len(respons2['objects'])
        filer_i_dag = []
        for i in range(antal):
            utfil = hent_filar_web_logs(respons2['objects'][i]['id'], i)
            filer_i_dag.append(utfil)
    else:
        print(f"Feil i spørjing, status {r.status_code}")
except Exception as e:
    print(f"Noko gjekk gale: {e}")

Sender inkrementell spørjing for dei siste 4 timane til https://api-gateway.instructure.com/dap/query/canvas_logs/table/web_logs/data
Sjekker status på jobb dc36ce22-d9c3-472f-971c-279d2660ea71
{'id': 'dc36ce22-d9c3-472f-971c-279d2660ea71', 'status': 'running', 'expires_at': '2025-10-02T06:25:35Z'}
Sjekker status på jobb dc36ce22-d9c3-472f-971c-279d2660ea71
{'id': 'dc36ce22-d9c3-472f-971c-279d2660ea71', 'status': 'running', 'expires_at': '2025-10-02T06:25:35Z'}
Sjekker status på jobb dc36ce22-d9c3-472f-971c-279d2660ea71
{'id': 'dc36ce22-d9c3-472f-971c-279d2660ea71', 'status': 'running', 'expires_at': '2025-10-02T06:25:35Z'}
Sjekker status på jobb dc36ce22-d9c3-472f-971c-279d2660ea71
{'id': 'dc36ce22-d9c3-472f-971c-279d2660ea71', 'status': 'running', 'expires_at': '2025-10-02T06:25:35Z'}
Sjekker status på jobb dc36ce22-d9c3-472f-971c-279d2660ea71
{'id': 'dc36ce22-d9c3-472f-971c-279d2660ea71', 'status': 'running', 'expires_at': '2025-10-02T06:25:35Z'}
Sjekker status på jobb dc36ce22-d9c3

## 2. Lese inn og analysere data
Datafilene eg får frå web_logs *kan* vere veldig store, og inneheld veldig mange felt. Det kan løne seg å lagre fila som tekstfil, og så heller lese den inn att til pandas, med berre dei data eg er interessert i.

Ein ting kan vere å hente inn brukar og kva url dei har vore inne på. Og så tar eg vekk dei som ikkje er "ekte" studentar (eg har til dømes 1 % av all aktivitet i CCanvas ...)

In [5]:
urls = pd.read_csv(utfil, sep=',', usecols=['value.user_id', 'value.url'])

In [10]:
reelle_urls = urls[~urls['value.user_id'].isin([2916, 12477])]

In [11]:
reelle_urls['value.user_id'].value_counts()

value.user_id
94594.0     271
93489.0     270
114594.0    270
93058.0     255
109803.0    178
           ... 
80904.0       1
114738.0      1
81747.0       1
119121.0      1
101619.0      1
Name: count, Length: 926, dtype: int64

In [25]:
sider = reelle_urls[reelle_urls['value.url'].str.contains('/pages/')]

In [22]:
reelle_urls[reelle_urls['value.url'].str.contains('/pages/')].to_csv('sider.csv', index=False)

In [23]:
digitalundervising = reelle_urls[reelle_urls['value.url'].str.contains('corses/12436/pages/')]

In [27]:
sider

,value.user_id,value.url
5,105484.0,/api/v1/courses/33227/pages/hjem-side-mal/revi...
11,30944.0,/api/v1/courses/31437/pages/videoer-til-kapitt...
25,102899.0,/api/v1/courses/31419/pages/konsumentteori-del...
31,87884.0,/api/v1/courses/33507/pages/retningslinjer-for...
46,98746.0,/api/v1/courses/31508/pages/faginformasjon/rev...
...,...,...
16416,30944.0,/api/v1/courses/31437/pages/videoer-til-kapitt...
16429,NaN,/courses/33249/pages/tema-7-is-pc-mp?module_it...
16439,2255.0,/api/v1/courses/29419/pages/fos-8140-innforing...
16447,70212.0,/api/v1/courses/31664/pages/uke-40-derivasjon/...


In [28]:
for i, r in sider.iterrows():
    print(i, r['value.user_id'], r['value.url'])

5 105484.0 /api/v1/courses/33227/pages/hjem-side-mal/revisions/latest?summary=true
11 30944.0 /api/v1/courses/31437/pages/videoer-til-kapittel-2/revisions/latest?summary=true
25 102899.0 /api/v1/courses/31419/pages/konsumentteori-del-1-videoer/revisions/latest?summary=true
31 87884.0 /api/v1/courses/33507/pages/retningslinjer-for-bacheloroppgaven/revisions/latest?summary=true
46 98746.0 /api/v1/courses/31508/pages/faginformasjon/revisions/latest?summary=true
60 115099.0 /api/v1/courses/31610/pages/forside/revisions/latest?summary=true
62 80930.0 /api/v1/courses/31945/pages/course-plan/revisions/latest?summary=true
67 115718.0 /api/v1/courses/32429/pages/2-dot-6-phrases-in-english-video-lecture/revisions/latest?summary=true
83 120473.0 /api/v1/courses/32896/pages/tips-til-problemformulering-ola-3/revisions/latest?summary=true
106 115099.0 /api/v1/courses/31610/pages/uke-35-2/revisions/latest?summary=true
111 114375.0 /api/v1/courses/32420/pages/welcome-to-enb803n/revisions/latest?summar

In [29]:
no

datetime.datetime(2025, 10, 1, 8, 25, 34, 742737)

In [31]:
no.isoformat()[0:16]

'2025-10-01T08:25'

In [37]:
sist_oppdatert[11:16]

'04:25'

In [34]:
print(f"Sidevisingar mellom {sist_oppdatert[0:16]} og {no.isoformat()[0:16]}: {len(sider)}")

Sidevisingar mellom 2025-10-01T04:25 og 2025-10-01T08:25: 1068


In [36]:
len(digitalundervising)

0